#### Load train data 

In [1]:
import pandas as pd 

df = pd.read_csv("merged_final_seq.csv")

In [2]:
df.head()

,accession,tax_id,term,aspect,protein_name,organism,sequence,length
0,A0A0C5B5G6,9606,GO:0001649,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
1,A0A0C5B5G6,9606,GO:0033687,P,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
2,A0A0C5B5G6,9606,GO:0005615,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
3,A0A0C5B5G6,9606,GO:0005634,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16
4,A0A0C5B5G6,9606,GO:0005739,C,MOTSC,HUMAN,MRWQEMGYIFYPRKLR,16


In [3]:
check_conflicts = (
    df.groupby("accession")["protein_name"]
    .nunique()
    .reset_index(name="unique_protein_name")
)

conflicts = check_conflicts[check_conflicts["unique_protein_name"] > 1]

if conflicts.empty:
    print("Each accession has only one protein name")
else:
    print(f"The following accession have conflicting protein names: \n {conflicts}")

Each accession has only one protein name


In [4]:
check_conflicts = (
    df.groupby("accession")["aspect"]
    .nunique()
    .reset_index(name="unique_aspect")
)

conflicts = check_conflicts[check_conflicts["unique_aspect"] > 1]

if conflicts.empty:
    print("Each accession has only one unique_aspect name")
else:
    print(f"The following accession have conflicting unique_aspect names: \n {conflicts}")

The following accession have conflicting unique_aspect names: 
         accession  unique_aspect
5      A0A023FFD0              2
9      A0A023I7E1              2
12     A0A024RBG1              2
14     A0A026W182              2
15     A0A044RE18              2
...           ...            ...
82399      X2JI34              2
82400      X4Y2L4              2
82401      X5JA13              3
82402      X5JB51              3
82403      X5M5N0              2

[59072 rows x 2 columns]


In [5]:
check_conflicts = (
    df.groupby("accession")["organism"]
    .nunique()
    .reset_index(name="unique_organism")
)

conflicts = check_conflicts[check_conflicts["unique_organism"] > 1]

if conflicts.empty:
    print("Each accession has only one unique_organism name")
else:
    print(f"The following accession have conflicting unique_organism names: \n {conflicts}")

Each accession has only one unique_organism name


#### Feature Engineering: 

#### Grouped accession based on term 

In [6]:
grouped_df = (df.groupby("accession")["term"]
              .apply(list)
              .reset_index(name="go_terms")
              )

grouped_df.head()

,accession,go_terms
0,A0A023FBW4,[GO:0019958]
1,A0A023FBW7,[GO:0019957]
2,A0A023FDY8,[GO:0019957]
3,A0A023FF81,[GO:0019958]
4,A0A023FFB5,[GO:0019957]


In [7]:
grouped_df.shape

(82404, 2)

In [8]:
seq_df = df[["accession", "protein_name", "sequence", "length", "organism"]].drop_duplicates()
final_df = grouped_df.merge(seq_df, on="accession", how="left")
final_df.head()

,accession,go_terms,protein_name,sequence,length,organism
0,A0A023FBW4,[GO:0019958],E1142,MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,AMBCJ
1,A0A023FBW7,[GO:0019957],EV546,MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,AMBCJ
2,A0A023FDY8,[GO:0019957],EV974,MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,AMBCJ
3,A0A023FF81,[GO:0019958],E1126,MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,AMBCJ
4,A0A023FFB5,[GO:0019957],EV983,MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,AMBCJ


In [9]:
seq_df.shape

(82404, 5)

In [10]:
final_df.shape

(82404, 6)

#### Apply Embedding on the sequence column 

In [11]:
from transformers import T5Tokenizer, T5EncoderModel
import torch
import numpy as np

# Load model and move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("abc",device)

tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50")
model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_uniref50").to(device)
model = model.eval()

def get_embedding(sequence):
    # Clean up: remove spaces and make uppercase
    sequence = " ".join(list(sequence))
    ids = tokenizer(sequence, return_tensors="pt", add_special_tokens=True, truncation=True)
    ids = {k: v.to(device) for k, v in ids.items()}
    
    with torch.no_grad():
        embedding = model(**ids).last_hidden_state.mean(dim=1).cpu().numpy()
    return embedding


c:\Users\najaa\Desktop\CAFA 6\cafa-6-protein-function-prediction\py312-bio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


abc cuda


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [12]:
import gc
def get_embedding_optimized(sequence, tokenizer, model, device, max_length=512):
    """
    Generate protein embedding with memory optimization
    
    Args:
        sequence: protein sequence string
        tokenizer: T5Tokenizer
        model: T5EncoderModel
        device: torch device
        max_length: maximum sequence length (default 512, adjust based on GPU memory)
    """
    # Truncate very long sequences
    if len(sequence) > max_length:
        sequence = sequence[:max_length]
    
    # Add spaces between amino acids (required for ProtT5)
    sequence = ' '.join(list(sequence))
    
    # Tokenize with explicit max_length
    ids = tokenizer(
        sequence,
        add_special_tokens=True,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    
    # Move to device
    ids = {k: v.to(device) for k, v in ids.items()}
    
    # Generate embedding with memory management
    with torch.no_grad():
        embedding = model(**ids).last_hidden_state
        # Use only non-padding tokens for mean pooling
        attention_mask = ids['attention_mask']
        mask_expanded = attention_mask.unsqueeze(-1).expand(embedding.size()).float()
        sum_embeddings = torch.sum(embedding * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_embedding = (sum_embeddings / sum_mask).cpu().numpy()
    
    # Clear GPU memory
    del ids, embedding, attention_mask, mask_expanded
    torch.cuda.empty_cache()
    
    return mean_embedding[0]

In [13]:
import tqdm 
import gc


def generate_embeddings_batch(df, batch_size=4, max_length=512, use_cpu_fallback=True):
    """
    Generate embeddings for all sequences with batching and memory optimization
    
    Args:
        df: DataFrame with 'sequence' column
        batch_size: number of sequences to process at once
        max_length: maximum sequence length
        use_cpu_fallback: if True, use CPU for very long sequences
    """
    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load model and tokenizer
    print("Loading ProtT5 model...")
    tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc", do_lower_case=False)
    model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc")
    
    # Use half precision to save memory (if on GPU)
    if device.type == 'cuda':
        model = model.half()
    
    model = model.to(device)
    model.eval()
    
    embeddings = []
    
    # Sort sequences by length to batch similar lengths together
    df_sorted = df.copy()
    df_sorted['seq_length'] = df_sorted['sequence'].str.len()
    df_sorted = df_sorted.sort_values('seq_length')
    original_indices = df_sorted.index.tolist()
    
    print(f"Processing {len(df_sorted)} sequences...")
    
    for i in tqdm.tqdm(range(0, len(df_sorted), batch_size)):
        batch_seqs = df_sorted['sequence'].iloc[i:i+batch_size].tolist()
        batch_embeddings = []
        
        for seq in batch_seqs:
            try:
                # Check if sequence is too long and use CPU if needed
                if len(seq) > max_length * 2 and use_cpu_fallback and device.type == 'cuda':
                    # Move model to CPU temporarily for this sequence
                    model_cpu = model.cpu()
                    emb = get_embedding_optimized(seq, tokenizer, model_cpu, 
                                                 torch.device('cpu'), max_length)
                    model = model.to(device)
                    torch.cuda.empty_cache()
                else:
                    emb = get_embedding_optimized(seq, tokenizer, model, device, max_length)
                
                batch_embeddings.append(emb)
                
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"\nOOM error for sequence length {len(seq)}, using CPU fallback...")
                    torch.cuda.empty_cache()
                    gc.collect()
                    
                    # Fallback to CPU
                    model_cpu = model.cpu()
                    emb = get_embedding_optimized(seq, tokenizer, model_cpu, 
                                                 torch.device('cpu'), max_length//2)
                    model = model.to(device)
                    batch_embeddings.append(emb)
                else:
                    raise e
        
        embeddings.extend(batch_embeddings)
        
        # Clear cache after each batch
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()
    
    # Restore original order
    embeddings_reordered = [None] * len(embeddings)
    for i, orig_idx in enumerate(original_indices):
        embeddings_reordered[orig_idx] = embeddings[i]
    
    # Convert to DataFrame
    embedding_dim = len(embeddings_reordered[0])
    embedding_df = pd.DataFrame(
        embeddings_reordered, 
        columns=[f'emb_{i}' for i in range(embedding_dim)]
    )
    
    print(f"Generated embeddings with shape: {embedding_df.shape}")
    embedding_df.to_csv("embedding.csv", index=False)

    return embedding_df

In [14]:
embedding_df = generate_embeddings_batch(final_df)

Using device: cuda
Loading ProtT5 model...
Processing 82404 sequences...


  0%|          | 1/20601 [00:11<65:41:44, 11.48s/it]


KeyboardInterrupt: 

In [ ]:
# embeddings = []

# for seq in  final_df['sequence']: 
#     emb = get_embedding(seq)
#     embeddings.append(emb)

# embedding_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(len(embeddings[0]))])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.42 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 15.58 GiB is allocated by PyTorch, and 618.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

#### Apply one-hot encoding for taxonomy features 'organism'